### Importação das Bibliotecas Necessárias

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal

plt.rcParams['figure.figsize'] = [10, 6]
plt.rcParams['axes.grid'] = True

---
### **Questão 1: Emulação de Interferência Periódica na Rede de Sensores**
*Gere um sinal composto por duas senoides de frequências diferentes. Projetar um filtro passa-baixa e verificar quais componentes permanecem após a filtragem.*

In [ ]:
fs = 1000  # Frequência de amostragem (Hz)
t = np.arange(0, 1.0, 1/fs)

# Sinal agrícola lento (Ex: variação de umidade, 2 Hz) + ruído acoplado da rede elétrica (60 Hz)
f_sinal, f_ruido = 2, 60
sinal_agricola = np.sin(2 * np.pi * f_sinal * t) + 0.5 * np.sin(2 * np.pi * f_ruido * t)

# Filtro Passa-Baixa Butterworth para isolar a variação real da planta
fcorte = 15
b, a = signal.butter(4, fcorte, fs=fs, btype='low')
sinal_filtrado = signal.filtfilt(b, a, sinal_agricola)

plt.figure()
plt.plot(t, sinal_agricola, label='Leitura Bruta com Acoplamento (2Hz + 60Hz)')
plt.plot(t, sinal_filtrado, 'g', linewidth=2.5, label='Variável Agrícola Filtrada (Passa-Baixa)')
plt.title('Q1: Remoção de Interferência Periódica de Rede Elétrica em Sensor')
plt.xlabel('Tempo (s)')
plt.ylabel('Amplitude')
plt.legend()
plt.show()

---
### **Questão 2: Atenuação de Ruído Térmico Eletrônico com Filtro FIR**
*Gerar um sinal contaminado por ruído branco aditivo. Projetar um filtro FIR passa-baixa e avaliar sua capacidade de reduzir o ruído.*

In [ ]:
sinal_puro = np.sin(2 * np.pi * 3 * t)  # Sinal limpo do sensor
ruido_termico = np.random.normal(0, 0.6, len(t))  # Ruído do circuito eletrônico
sinal_ruidoso = sinal_puro + ruido_termico

# Filtro FIR via Janela de Hamming
numtaps = 51
b_fir = signal.firwin(numtaps, 10, fs=fs)
sinal_filtrado_fir = signal.lfilter(b_fir, 1, sinal_ruidoso)

plt.figure()
plt.plot(t, sinal_ruidoso, alpha=0.5, label='Sensor com Ruído Térmico')
plt.plot(t, sinal_filtrado_fir, 'r', linewidth=2, label='Filtrado com FIR (Fase Linear)')
plt.title('Q2: Filtragem FIR de Ruído Térmico de Circuitos de Aquisição')
plt.xlabel('Tempo (s)')
plt.ylabel('Amplitude')
plt.legend()
plt.show()

---
### **Questão 3: Análise de Custo-Benefício Computacional: Abordagem IIR**
*Repetir o experimento anterior utilizando um filtro IIR Butterworth. Compare os resultados obtidos com o filtro FIR.*

In [ ]:
b_iir, a_iir = signal.butter(3, 10, fs=fs, btype='low')  # Apenas ordem 3
sinal_filtrado_iir = signal.lfilter(b_iir, a_iir, sinal_ruidoso)

plt.figure()
plt.plot(t[:500], sinal_ruidoso[:500], alpha=0.3, label='Leitura Bruta')
plt.plot(t[:500], sinal_filtrado_fir[:500], 'r', label='FIR (51 coeficientes)')
plt.plot(t[:500], sinal_filtrado_iir[:500], 'b', label='IIR Butterworth (Ordem 3)')
plt.title('Q3: FIR vs IIR na Suavização de Sinais Agrícolas')
plt.xlabel('Tempo (s)')
plt.ylabel('Amplitude')
plt.legend()
plt.show()

---
### **Questão 4: Comparação das Respostas em Magnitude (Bode)**
*Projetar filtros passa-baixa FIR e IIR com frequências de corte semelhantes. Compare as respostas em frequência e discuta as diferenças observadas.*

In [ ]:
w_fir, h_fir = signal.freqz(b_fir, 1, fs=fs)
w_iir, h_iir = signal.freqz(b_iir, a_iir, fs=fs)

plt.figure()
plt.plot(w_fir, 20 * np.log10(np.abs(h_fir)), 'r', label='FIR (Janela de Hamming)')
plt.plot(w_iir, 20 * np.log10(np.abs(h_iir)), 'b', label='IIR Butterworth')
plt.axvline(10, color='k', linestyle='--', label='Corte do Canal (10 Hz)')
plt.title('Q4: Resposta em Magnitude das Soluções de Filtragem')
plt.xlabel('Frequência (Hz)')
plt.ylabel('Atenuação (dB)')
plt.ylim([-60, 5])
plt.legend()
plt.show()

----- 
### **Questão 5: Estabilidade Estrutural do Filtro IIR no Plano Z**
*Utilize ferramentas computacionais para representar os polos e zeros de um filtro IIR. Discuta a relação entre sua localização e a estabilidade do sistema.*

In [ ]:
z, p, k = signal.tf2zpk(b_iir, a_iir)

plt.figure(figsize=(6,6))
circulo = plt.Circle((0,0), 1, fill=False, color='gray', linestyle='--')
plt.gca().add_patch(circulo)

plt.scatter(np.real(z), np.imag(z), marker='o', edgecolors='b', s=100, label='Zeros')
plt.scatter(np.real(p), np.imag(p), marker='x', color='r', s=100, label='Polos')

plt.axhline(0, color='black', lw=0.5)
plt.axvline(0, color='black', lw=0.5)
plt.title('Q5: Diagrama de Estabilidade de Polos e Zeros')
plt.xlim([-1.2, 1.2])
plt.ylim([-1.2, 1.2])
plt.legend()
plt.gca().set_aspect('equal')
plt.show()

---
### **Questão 6: Comportamento Transiente e Resposta ao Impulso**
*Analise a resposta ao impulso de um filtro FIR e de um filtro IIR. Explique por que um apresenta resposta finita e o outro resposta infinita.*

In [ ]:
impulso = np.zeros(80)
impulso[0] = 1.0

h_fir = signal.lfilter(b_fir, 1, impulso)
h_iir = signal.lfilter(b_iir, a_iir, impulso)

plt.figure()
plt.subplot(2, 1, 1)
plt.stem(h_fir, linefmt='r-', markerfmt='ro')
plt.title('Q6: Resposta ao Impulso - Sistema FIR (Estável e Finito)')

plt.subplot(2, 1, 2)
plt.stem(h_iir, linefmt='b-', markerfmt='bo')
plt.title('Resposta ao Impulso - Sistema IIR (Infinito Amortecido)')
plt.xlabel('Amostras')
plt.tight_layout()
plt.show()

---
### **Questão 7: Isolamento de Vibração Mecânica de Maquinário Agrícola via Passa-Faixa**
*Projetar um filtro passa-faixa capaz de selecionar uma frequência específica presente em um sinal composto. Verifique o resultado no domínio da frequência.*

In [ ]:
# Um sensor de vibração no trator capta: oscilação do terreno (5Hz), vibração do motor (45Hz) e harmônicas (120Hz)
sinal_vibração = np.sin(2 * np.pi * 5 * t) + np.sin(2 * np.pi * 45 * t) + np.sin(2 * np.pi * 120 * t)

# Filtro Passa-Faixa para isolar exclusivamente a vibração nominal do motor (40 Hz a 50 Hz)
b_pf, a_pf = signal.butter(4, [38, 52], fs=fs, btype='bandpass')
sinal_filtrado_pf = signal.filtfilt(b_pf, a_pf, sinal_vibração)

freqs = np.fft.rfftfreq(len(t), 1/fs)
fft_bruta = np.abs(np.fft.rfft(sinal_vibração))
fft_filtrada = np.abs(np.fft.rfft(sinal_filtrado_pf))

plt.figure()
plt.plot(freqs, fft_bruta, label='Vibração Total Captada')
plt.plot(freqs, fft_filtrada, 'g', linewidth=2.5, label='Componente Isolada do Motor (45 Hz)')
plt.title('Q7: Análise Espectral para Diagnóstico de Falhas em Maquinário por Filtro Passa-Faixa')
plt.xlabel('Frequência (Hz)')
plt.ylabel('Magnitude')
plt.xlim([0, 150])
plt.legend()
plt.show()

---
### **Questão 8: Integridade da Forma de Onda (Preservação de Fase)**
*Analise a resposta de fase de um filtro FIR e compare com a resposta de fase de um filtro IIR. Discuta o conceito de fase linear.*

In [ ]:
w_fir, h_fir = signal.freqz(b_fir, 1, fs=fs)
w_iir, h_iir = signal.freqz(b_iir, a_iir, fs=fs)

plt.figure()
plt.plot(w_fir, np.unwrap(np.angle(h_fir)), 'r', label='Fase Linear (FIR)')
plt.plot(w_iir, np.unwrap(np.angle(h_iir)), 'b', label='Fase Não-Linear (IIR)')
plt.title('Q8: Impacto do Deslocamento de Fase nos Sinais de Campo')
plt.xlabel('Frequência (Hz)')
plt.ylabel('Fase Desenrolada (rad)')
plt.legend()
plt.show()

---
### **Questão 9: Atraso de Grupo e Latência no Sistema IoT Agrícola**
*Calcule e compare o atraso de grupo de diferentes filtros digitais. Explique a importância desse parâmetro em sistemas de comunicação.*

In [ ]:
w1, gd_fir = signal.group_delay((b_fir, 1), w=fs)
w2, gd_iir = signal.group_delay((b_iir, a_iir), w=fs)

plt.figure()
plt.plot(w1, gd_fir, 'r', label='Atraso Constante (FIR)')
plt.plot(w2, gd_iir, 'b', label='Atraso Variável (IIR)')
plt.title('Q9: Análise do Atraso de Grupo (*Group Delay*)')
plt.xlabel('Frequência (Hz)')
plt.ylabel('Atraso (Amostras)')
plt.legend()
plt.show()

---
### **Questão 10: Validação de Caso Prático: Monitoramento de Umidade do Solo**
*Pesquise e implemente uma aplicação prática envolvendo filtragem digital, como remoção de ruído em áudio, suavização de sinais de sensores ou filtragem de vibrações mecânicas.*

In [ ]:
# Simulação de sinal de umidade volumétrica do solo (VWC) com picos espúrios gerados por instabilidade elétrica temporária
dados_reais_umidade = 35 - 5 * t  # Secagem gradual do solo ao longo do dia
picos_ruido = np.random.normal(0, 0.4, len(t))
picos_ruido[::40] += np.random.choice([-4, 4], size=len(t)//40) # Adiciona ruído impulsivo (espúrio de comutação)
leitura_sensor_agri = dados_reais_umidade + picos_ruido

# Filtro de Média Móvel Exponencial ou Janela Deslizante de 25 pontos
janela = 25
b_suave = np.ones(janela) / janela
dados_higienizados = signal.filtfilt(b_suave, 1, leitura_sensor_agri)

plt.figure()
plt.plot(t, leitura_sensor_agri, alpha=0.4, color='orange', label='Leitura Crua Contaminada')
plt.plot(t, dados_higienizados, 'darkblue', linewidth=2.5, label='Sinal Higienizado para Tomada de Decisão')
plt.plot(t, dados_reais_umidade, 'g--', label='Tendência Real de Secagem')
plt.title('Q10: Filtro de Suavização Aplicado a Dados de Umidade do Solo')
plt.xlabel('Tempo (Proporcional a Horas)')
plt.ylabel('Umidade Volumétrica (%)')
plt.legend()
plt.show()